# **Predicción de Citas Médicas**

### Prototipo demostrativo | IPS Salud a Tu Lado

Esta aplicación permite explorar las predicciones de asistencia, incumplimiento y cancelación de citas médicas mediante un modelo de Machine Learning.

*Herramienta desarrollada con fines académicos. Sus predicciones tienen limitaciones de precisión y no deben utilizarse como único criterio para tomar decisiones operativas.*

Integrantes
* Paola Andrea Duque Zuluaga
* Katerin Giraldo Betancur
* hon Edisson Hurtado Ocampo

In [1]:
# Librerías para la carga del modelo
import pickle
import numpy as np
import pandas as pd

# Transformaciones y codificación
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder

# Modelo predictivo
from sklearn.ensemble import GradientBoostingClassifier

# Pipeline y técnicas de balanceo
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTENC

# Carga de archivos en Colab
from google.colab import files

In [2]:
# Estrategia de undersampling: reducir ACTIVO al 50 %
def estrategia_undersampling(Y_train):
    cantidad_activos = (Y_train == 0).sum()
    return {0: int(cantidad_activos * 0.5)}


# Estrategia de oversampling: duplicar CITA CANCELADA
def estrategia_balanceo(Y_train):
    cantidad_canceladas = (Y_train == 1).sum()
    return {1: cantidad_canceladas * 2}

In [3]:
# Se carga el modelo
filename = 'modelo_citas_medicas.pkl'
modeloGB, labelencoder, variables = pickle.load(open(filename, 'rb'))

In [4]:
# Verificar los componentes recuperados
print("Modelo:", type(modeloGB).__name__)
print("Clases:", labelencoder.classes_)
print("Variables predictoras:", len(variables))

Modelo: Pipeline
Clases: ['ACTIVO' 'CITA CANCELADA' 'INCUMPLIDO']
Variables predictoras: 16


In [6]:
# Cargar los datos futuros para realizar predicciones
datos_futuros = pd.read_csv('citas_medicas_datos_futuros.csv')

# Visualizar los primeros cinco registros
datos_futuros.head()

,oportunidad,sexopaciente,programacronico,funcionalidad,rotulo,municipiosede,turno_doble,poblacion_vulnerable,grupo_convenionombre,grupo_municipio,dia_semana,mes,grupo_horacita,grupo_edad,grupo_pym,grupo_codigocups
0,8,M,Ninguno,TERAPIAS,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,JUEVES,SEPTIEMBRE,TARDE,PRIMERA_INFANCIA,CITA MORBILIDAD,TERAPIAS Y REHABILITACION
1,80,F,Ninguno,PROCEDIMIENTO,PROCEDIMIENTOS DE ELECTROCARDIOGRAMA,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MARTES,SEPTIEMBRE,MANIANA,VEJEZ,"PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES",CONSULTAS CLINICAS Y OTROS PROFESIONALES
2,4,F,Ninguno,CONSULTA,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MARTES,JULIO,MADRUGADA,JUVENTUD,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES
3,0,F,Ninguno,CONSULTA,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MIERCOLES,JULIO,MANIANA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES
4,16,F,Ninguno,CONSULTA,SIN ROTULO,CAIMITO,NO,Ninguna,Nueva EPS Subsidiado,0 a 1 horas,VIERNES,JULIO,MEDIODIA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES


## Predicciones

In [7]:
# Predecir el estado de las citas médicas
predicciones = modeloGB.predict(datos_futuros)
print(predicciones)
# Convertir las predicciones a sus categorías originales
predicciones = labelencoder.inverse_transform(predicciones)

[2 0 0 0 1 0 0 0 0 0 0 1 2 0 0 0 2 2 0 0 0 0 1 2]


In [8]:
# Agregar las predicciones al DataFrame de datos futuros
datos_futuros['Prediccion'] = predicciones

# Visualizar los primeros cinco registros con su predicción
datos_futuros.head()

,oportunidad,sexopaciente,programacronico,funcionalidad,rotulo,municipiosede,turno_doble,poblacion_vulnerable,grupo_convenionombre,grupo_municipio,dia_semana,mes,grupo_horacita,grupo_edad,grupo_pym,grupo_codigocups,Prediccion
0,8,M,Ninguno,TERAPIAS,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,JUEVES,SEPTIEMBRE,TARDE,PRIMERA_INFANCIA,CITA MORBILIDAD,TERAPIAS Y REHABILITACION,INCUMPLIDO
1,80,F,Ninguno,PROCEDIMIENTO,PROCEDIMIENTOS DE ELECTROCARDIOGRAMA,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MARTES,SEPTIEMBRE,MANIANA,VEJEZ,"PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES",CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
2,4,F,Ninguno,CONSULTA,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MARTES,JULIO,MADRUGADA,JUVENTUD,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
3,0,F,Ninguno,CONSULTA,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MIERCOLES,JULIO,MANIANA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
4,16,F,Ninguno,CONSULTA,SIN ROTULO,CAIMITO,NO,Ninguna,Nueva EPS Subsidiado,0 a 1 horas,VIERNES,JULIO,MEDIODIA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,CITA CANCELADA


In [9]:
# Mostrar los datos futuros con sus predicciones finales
datos_futuros

,oportunidad,sexopaciente,programacronico,funcionalidad,rotulo,municipiosede,turno_doble,poblacion_vulnerable,grupo_convenionombre,grupo_municipio,dia_semana,mes,grupo_horacita,grupo_edad,grupo_pym,grupo_codigocups,Prediccion
0,8,M,Ninguno,TERAPIAS,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,JUEVES,SEPTIEMBRE,TARDE,PRIMERA_INFANCIA,CITA MORBILIDAD,TERAPIAS Y REHABILITACION,INCUMPLIDO
1,80,F,Ninguno,PROCEDIMIENTO,PROCEDIMIENTOS DE ELECTROCARDIOGRAMA,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MARTES,SEPTIEMBRE,MANIANA,VEJEZ,"PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES",CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
2,4,F,Ninguno,CONSULTA,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MARTES,JULIO,MADRUGADA,JUVENTUD,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
3,0,F,Ninguno,CONSULTA,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MIERCOLES,JULIO,MANIANA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
4,16,F,Ninguno,CONSULTA,SIN ROTULO,CAIMITO,NO,Ninguna,Nueva EPS Subsidiado,0 a 1 horas,VIERNES,JULIO,MEDIODIA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,CITA CANCELADA
5,2,F,Ninguno,PROCEDIMIENTO,MONITOREO PRESION ARTERIAL,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,MIERCOLES,JULIO,MANIANA,VEJEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
6,1,M,Ninguno,CONSULTA,SIN ROTULO,BUENAVISTA,NO,Ninguna,Nueva EPS Subsidiado,0 a 1 horas,MIERCOLES,SEPTIEMBRE,TARDE,JUVENTUD,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
7,84,M,Ninguno,PROCEDIMIENTO,PROCEDIMIENTOS DE ELECTROCARDIOGRAMA,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,1 a 2 horas,JUEVES,JULIO,MADRUGADA,ADULTEZ,CITA MORBILIDAD,CONSULTAS CLINICAS Y OTROS PROFESIONALES,ACTIVO
8,0,F,Nefroproteccion,TRATAMIENTO ODONTOLOGICO,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Contributivo,0 a 1 horas,LUNES,AGOSTO,TARDE,VEJEZ,CITA MORBILIDAD,ODONTOLOGIA ESPECIALIZADA,ACTIVO
9,2,F,Ninguno,LABORATORIO,SIN ROTULO,SINCELEJO,NO,Ninguna,Nueva EPS Subsidiado,0 a 1 horas,VIERNES,SEPTIEMBRE,MADRUGADA,ADULTEZ,"PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES",LABORATORIO CLINICO,ACTIVO


## **Interfaz**

In [ ]:
# ENCABEZADO

In [ ]:
# Importar Streamlit
from PIL import Image
import streamlit as st


icono = Image.open("images/icono_corazon.png")

# Configurar la página
st.set_page_config(
    page_title="Predicción de Citas Médicas",
    page_icon=icono,
    layout="wide"
)

# Cargar los estilos CSS
with open('estilos.css', 'r', encoding='utf-8') as archivo:
    estilos = archivo.read()

st.markdown(f"<style>{estilos}</style>", unsafe_allow_html=True)

# Crear el encabezado de la aplicación
st.markdown("""
<div class="encabezado">
    <h1>Predicción de Citas Médicas</h1>
    <p>Prototipo demostrativo | IPS Salud a Tu Lado</p>
</div>
""", unsafe_allow_html=True)

2026-10-29 03:55:56.053 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 03:55:56.055 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 03:55:56.235 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-10-29 03:55:56.236 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 03:55:56.237 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 03:55:56.238 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-29 03:55:56.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

DeltaGenerator()

In [ ]:
# FORMULARIO

In [ ]:
# Sección 1. Información del paciente
st.subheader("Información del paciente")

col1, col2 = st.columns(2)

with col1:
    sexopaciente = st.selectbox("Sexo del paciente", ["F", "M"])

    grupo_edad = st.selectbox("Grupo de edad", ["ADOLESCENCIA", "ADULTEZ", "INFANCIA", "JUVENTUD", "PRIMERA_INFANCIA", "VEJEZ"])

    programacronico = st.selectbox( "Programa de atención de enfermedades crónicas",  ["Diabetes", "Hipertension", "Nefroproteccion", "Ninguno", "Sindrome Metabolico"])

with col2:
    poblacion_vulnerable = st.selectbox(
        "Población vulnerable",
        ["Adulta Mayor sin Protección", "Afrodescendiente",
         "Desplazada", "Indigena", "LGTBI", "Ninguna",
         "Poblacion Trabajo Sexual", "Poblacion a Cargo ICBF",
         "Situación de Calle", "Victima Conflicto Armado"]
    )

    grupo_municipio = st.selectbox("Tiempo estimado de desplazamiento a la sede", ["0 a 1 horas", "1 a 2 horas", "2 a 4 horas","4 a 6 horas", "mas de 6 horas"])

In [ ]:
# Sección 2. Información de la cita
st.subheader("Información de la cita")

col1, col2 = st.columns(2)

with col1:
    funcionalidad = st.selectbox("Tipo de atención", ["CONSULTA", "LABORATORIO", "PROCEDIMIENTO", "TERAPIAS", "TRATAMIENTO ODONTOLOGICO"])

    grupo_codigocups = st.selectbox(
        "Grupo de procedimiento o servicio",
        ["CONSULTAS CLINICAS Y OTROS PROFESIONALES",
         "LABORATORIO CLINICO",
         "ODONTOLOGIA ESPECIALIZADA",
         "ODONTOLOGIA GENERAL Y PREVENTIVA",
         "PATOLOGIA",
         "PROCEDIMIENTOS DE ENFERMERIA",
         "TERAPIAS Y REHABILITACION"]
    )

    grupo_pym = st.selectbox("Tipo de programa", ["CITA MORBILIDAD", "PROMOCION, PREVENCION Y PROGRAMAS ESPECIALES"])

with col2:
    grupo_convenionombre = st.selectbox( "Convenio o entidad responsable", ["Coosalud", "Nueva EPS Contributivo",
         "Nueva EPS Subsidiado", "Otros", "Particular",
         "Proteger EPS Contributivo", "Proteger EPS Subsidiado"]
    )

    municipiosede = st.selectbox( "Municipio de la sede", ["BUENAVISTA", "CAIMITO", "COROZAL", "GUARANDA",
         "MAJAGUAL", "SAMPUES", "SAN ANTONIO DE PALMITO",
         "SAN BENITO", "SAN LUIS DE SINCE", "SAN MARCOS",
         "SAN PEDRO", "SANTIAGO DE TOLU", "SINCELEJO", "SUCRE"]
    )

    rotulo = st.selectbox("Rótulo del servicio",
        ["AGENDAMIENTO LABORATORIO CLINICO",
         "AGENDAMIENTO PATOLOGIA", "CAMBIO DE SONDA",
         "CONSULTA PRIMERA VEZ ODONTOLOGIA",
         "CONSULTA TRATAMIENTO ODONTOLOGIA",
         "MONITOREO FETAL", "MONITOREO PRESION ARTERIAL",
         "PROCEDIMIENTOS COLPOSCOPIA",
         "PROCEDIMIENTOS DE ELECTROCARDIOGRAMA",
         "PROCEDIMIENTOS MENORES ENFERMERIA",
         "SIN ROTULO", "TOMA DE MUESTRAS DOMICILIARIAS"]
    )

In [ ]:
# Sección 3. Programación de la cita
st.subheader("Programación de la cita")

col1, col2 = st.columns(2)

with col1:
    oportunidad = st.slider(
    "Oportunidad de la cita (días)",
    min_value=0,
    max_value=150,
    value=0,
    step=1
    )

    dia_semana = st.selectbox("Día de la semana",["DOMINGO", "JUEVES", "LUNES", "MARTES","MIERCOLES", "SABADO", "VIERNES"])

    mes = st.selectbox("Mes de la cita", ["AGOSTO", "JULIO", "SEPTIEMBRE"])

with col2:
    grupo_horacita = st.selectbox("Horario de la cita", ["MADRUGADA", "MANIANA", "MEDIODIA", "NOCHE", "TARDE"])

    turno_doble = st.selectbox("Turno doble",["NO", "SI"])

In [ ]:
# Preparar los datos ingresados en el formulario
datos = pd.DataFrame([{
    'oportunidad': oportunidad,
    'sexopaciente': sexopaciente,
    'programacronico': programacronico,
    'funcionalidad': funcionalidad,
    'rotulo': rotulo,
    'municipiosede': municipiosede,
    'turno_doble': turno_doble,
    'poblacion_vulnerable': poblacion_vulnerable,
    'grupo_convenionombre': grupo_convenionombre,
    'grupo_municipio': grupo_municipio,
    'dia_semana': dia_semana,
    'mes': mes,
    'grupo_horacita': grupo_horacita,
    'grupo_edad': grupo_edad,
    'grupo_pym': grupo_pym,
    'grupo_codigocups': grupo_codigocups
}])

# Mantener el orden de las variables del modelo
datos = datos[variables]

In [ ]:
# Ejecutar la predicción cuando el usuario presione el botón
if st.button("🔍 Predecir estado de la cita"):

    # Obtener la predicción del modelo
    prediccion = modeloGB.predict(datos)

    # Convertir la predicción a su categoría original
    estado_predicho = labelencoder.inverse_transform(prediccion)[0]

    # Agregar la predicción a los datos ingresados
    resultado = datos.copy()
    resultado["Predicción"] = estado_predicho

    # Mostrar el resultado de la predicción
    st.subheader("Resultado de la predicción")
    st.success(f"Estado predicho de la cita: {estado_predicho}")

    # Mostrar los datos utilizados para la predicción
    st.markdown("**Datos utilizados para la predicción**")
    st.dataframe(datos, hide_index=True, use_container_width=True)


In [ ]:
# Informar sobre el desempeño del modelo
st.warning(
    "El modelo obtuvo una exactitud del 58,73 % en los datos de prueba. "
    "Las predicciones son orientativas y deben interpretarse con precaución."
)